In [1]:
from sentence_transformers import SentenceTransformer
import torch
from langchain_google_genai import ChatGoogleGenerativeAI

c:\Users\ADMIN\DensoMind\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
API_KEY = "YOUR_GOOGLE_API_KEY_HERE"
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=API_KEY,
    temperature=0.2
)

In [42]:
model.invoke("hi")

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--35a253a7-882b-4e51-8b19-54ec6b55412c-0', usage_metadata={'input_tokens': 2, 'output_tokens': 251, 'total_tokens': 253, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 241}})

In [125]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from pydantic import BaseModel, Field
from typing import List

In [126]:
class QuizQuestion(BaseModel):
    question: str = Field(description="Nội dung câu hỏi")
    options: List[str] = Field(description="Danh sách 4 đáp án lựa chọn có đánh dấu A, B, C, D")
    correct_answer: str = Field(description="Đáp án đúng (phải nằm trong danh sách options)")
    explanation: str = Field(description="Giải thích ngắn gọn tại sao đáp án này đúng")

class Quiz(BaseModel):
    questions: List[QuizQuestion]


parser = JsonOutputParser(pydantic_object=Quiz)

In [127]:
template = """
Bạn là một chuyên gia đào tạo kỹ thuật trong nhà máy. 
Nhiệm vụ của bạn là tạo ra một bài kiểm tra trắc nghiệm (Quiz) dựa trên nội dung văn bản được cung cấp.

Yêu cầu:
1. Tạo ra 3 câu hỏi trắc nghiệm.
2. Các câu hỏi phải tập trung vào các chi tiết kỹ thuật, an toàn hoặc quy trình quan trọng.
3. Ngôn ngữ: Tiếng Việt.
4. Định dạng đầu ra CHỈ LÀ JSON thuần túy, không có markdown ```json```.

Nội dung tài liệu:
{context}

{format_instructions}
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# 4. Tạo Chain (Dây chuyền xử lý)
chain = prompt | model | parser

In [128]:
chain

PromptTemplate(input_variables=['context'], input_types={}, partial_variables={'format_instructions': 'STRICT OUTPUT FORMAT:\n- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.\n- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).\n- Do not prepend or append any text (e.g., do not write "Here is the JSON:").\n- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.\n\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the outpu

In [145]:
context_text = """
Quy trình vận hành máy ép nhựa DENSO-X1:
Bước 1: Kiểm tra an toàn. Đảm bảo nút dừng khẩn cấp (E-Stop) đang ở trạng thái mở. Đeo kính bảo hộ và găng tay chịu nhiệt.
Bước 2: Khởi động nguồn. Bật công tắc chính ở phía sau máy. Đợi màn hình hiển thị "Ready".
Bước 3: Nạp nguyên liệu. Đổ hạt nhựa vào phễu nạp liệu. Lưu ý không đổ quá vạch MAX.
Bước 4: Vận hành. Nhấn nút START màu xanh. Nếu có đèn báo lỗi màu đỏ, nhấn nút E-Stop ngay lập tức.
"""

print("Đang suy nghĩ và tạo quiz...")

try:
    # Gọi Chain
    result = chain.invoke({"context": context_text})
    
    # print("✅ Kết quả trả về (Dạng Dictionary/JSON):")
    # import json
    # print(json.dumps(result, indent=2, ensure_ascii=False))
    
    # Test truy cập dữ liệu như code thật
    print("\n--- DEMO HIỂN THỊ TRÊN APP ---")
    # for i, q in enumerate(result['questions']):
    #     print(f"Câu {i+1}: {q['question']}")
    #     for opt in q['options']:
    #         print(f"  * {opt}")
    #     print(f"  => Đáp án đúng: {q['correct_answer']}")
    #     print("-" * 20)
    letters = ['A', 'B', 'C', 'D']
    score = 0
    questions = 0
    for i, q in enumerate(result['questions']):
        questions += 1
        print(f"Câu {i+1}: {q['question']}")
        for l, opt in enumerate(q['options']):
            print(f"    {opt}")
        while True:
            user_choice = input(f"Nhập các đáp án (A-D): ").upper().strip()
            if user_choice in letters:
                break
            print(f"Chỉ chọn A, B, C hoặc D.")
        selected_index = letters.index(user_choice)
        selected_text = q['options'][selected_index]

        is_correct = (selected_text == q['correct_answer'])
        if is_correct:
            print(f"Chính xác!!!!!!!")
            print(f"Đáp án : {q['correct_answer']}")
            print(f"Giải thích : {q['explanation']}")
            score += 1
        else:
            print(f"o hel na bru")
            print(f"Lựa chọn của bạn : {selected_text}")
            print(f"Lựa chọn chính xác : {q['correct_answer']}")
            print(f"Giải thích : {q['explanation']}")

    print(f"Số câu trả lời đúng : {score}/{questions}")
except Exception as e:
    print(f"Lỗi rồi: {e}")

Đang suy nghĩ và tạo quiz...

--- DEMO HIỂN THỊ TRÊN APP ---
Câu 1: Thiết bị bảo hộ cá nhân (PPE) nào được yêu cầu phải đeo trước khi vận hành máy ép nhựa DENSO-X1?
    A. Mũ bảo hiểm và giày bảo hộ
    B. Kính bảo hộ và găng tay chịu nhiệt
    C. Áo phản quang và nút bịt tai
    D. Mặt nạ phòng độc và ủng cao su
Chính xác!!!!!!!
Đáp án : B. Kính bảo hộ và găng tay chịu nhiệt
Giải thích : Bước 1 của quy trình vận hành máy ép nhựa DENSO-X1 yêu cầu 'Đeo kính bảo hộ và găng tay chịu nhiệt' để đảm bảo an toàn cho người vận hành.
Câu 2: Theo quy trình vận hành máy ép nhựa DENSO-X1, hành động nào cần thực hiện ngay lập tức nếu có đèn báo lỗi màu đỏ trong quá trình vận hành?
    A. Nhấn nút START màu xanh để khởi động lại.
    B. Tắt công tắc chính ở phía sau máy.
    C. Nhấn nút dừng khẩn cấp (E-Stop).
    D. Đổ thêm hạt nhựa vào phễu nạp liệu.
o hel na bru
Lựa chọn của bạn : D. Đổ thêm hạt nhựa vào phễu nạp liệu.
Lựa chọn chính xác : C. Nhấn nút dừng khẩn cấp (E-Stop).
Giải thích : Bước 4 c

In [65]:
prompt_rag = """
Bạn là một chuyên gia đào tạo kỹ thuật trong nhà máy.
Nhiệm vụ của bạn là trả lời các câu hỏi liên quan CHỈ DỰA VÀO nguồn tài liệu được cung cấp liên quan đến kỹ thuật vận hành máy móc trong nhà máy.
YÊU CẦU:
1. Trả lời chính xác dựa vào nguồn tại liệu được cung cấp.
2. Nếu không tìm thấy câu trả lời trong nguồn tài liệu, hãy trả lời 'Không biết'.
3. Thích ứng theo ngôn ngữ người hỏi, nếu người hỏi dùng tiếng Việt thì trả lời tiếng Việt, nếu người hỏi dùng tiếng Anh thì trả lời tiếng Anh.

Nội dung tài liệu:
{context}

Câu hỏi: 
{question}
"""

prompt_rag = PromptTemplate(
    template=prompt_rag,
    input_variables=["context", "question"]
)
chain_rag = prompt_rag | model | StrOutputParser()

In [62]:
context_text = """
Quy trình vận hành máy ép nhựa DENSO-X1:
Bước 1: Kiểm tra an toàn. Đảm bảo nút dừng khẩn cấp (E-Stop) đang ở trạng thái mở. Đeo kính bảo hộ và găng tay chịu nhiệt.
Bước 2: Khởi động nguồn. Bật công tắc chính ở phía sau máy. Đợi màn hình hiển thị "Ready".
Bước 3: Nạp nguyên liệu. Đổ hạt nhựa vào phễu nạp liệu. Lưu ý không đổ quá vạch MAX.
Bước 4: Vận hành. Nhấn nút START màu xanh. Nếu có đèn báo lỗi màu đỏ, nhấn nút E-Stop ngay lập tức.
"""

while True:
    user_question = input("Hỏi gì đi bạn ơi: ")
    if user_question.lower() in ['exit', 'quit', 'thoát']:
        break
    answer = chain_rag.invoke({
        "context": context_text,
        "question": user_question
    })
    print(f"Trả lời: {answer}")

Trả lời: Chào bạn, đây là quy trình vận hành máy ép nhựa DENSO-X1:

1.  **Kiểm tra an toàn:** Đảm bảo nút dừng khẩn cấp (E-Stop) đang ở trạng thái mở. Đeo kính bảo hộ và găng tay chịu nhiệt.
2.  **Khởi động nguồn:** Bật công tắc chính ở phía sau máy. Đợi màn hình hiển thị "Ready".
3.  **Nạp nguyên liệu:** Đổ hạt nhựa vào phễu nạp liệu. Lưu ý không đổ quá vạch MAX.
4.  **Vận hành:** Nhấn nút START màu xanh. Nếu có đèn báo lỗi màu đỏ, nhấn nút E-Stop ngay lập tức.


In [64]:
# toronto_accent_prompt = """
# you lowkey is a toronto accent expert, you speak some actual bullshit like "twos twos my word" or "top left", basically just talk like a true torontonian innit
# """

# toronto_accent_prompt = PromptTemplate(
#     template=toronto_accent_prompt,
#     input_variables=[]
# )
# toronto_chain = toronto_accent_prompt | model | StrOutputParser()
# while True:
#     user_input = input("Say something to the dumbass torontonian: ")
#     if user_input.lower() in ['exit', 'quit', 'bye']:
#         break
#     response = toronto_chain.invoke({})
#     print(response)

In [67]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [82]:
embedding_model = GoogleGenerativeAIEmbeddings(
            model = "models/text-embedding-004",
            google_api_key = API_KEY
        )
embedding_model

GoogleGenerativeAIEmbeddings(client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x000001A7EC08FF90>, async_client=None, model='models/text-embedding-004', task_type=None, google_api_key=SecretStr('**********'), credentials=None, client_options=None, base_url=None, transport=None, request_options=None)

In [71]:
test_chunks = [
    "Quy trình vận hành máy ép nhựa DENSO-X1 Bước 1: Kiểm tra nút dừng khẩn cấp (E-Stop).",
    "Lưu ý an toàn: Công nhân bắt buộc phải đeo kính bảo hộ và găng tay chịu nhiệt khi tiếp xúc khuôn đúc.",
    "Mã lỗi E05: Nhiệt độ khuôn quá cao. Cách xử lý: Tắt máy, kiểm tra hệ thống làm mát nước.",
    "Sơ đồ tổ chức xưởng A: Khu vực 1 là lắp ráp, Khu vực 2 là kiểm thử (QA/QC).",
    "Bảo trì định kỳ: Thay dầu máy mỗi 2000 giờ hoạt động hoặc 6 tháng một lần."
]

In [83]:
vectors = embedding_model.embed_documents(test_chunks)

In [85]:
sem_vectors = embedding_model.embed_documents(test_chunks, task_type="semantic_similarity")

In [77]:
len(vectors[0])

768

In [131]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_pymupdf4llm import PyMuPDF4LLMLoader
loader = PyPDFLoader('test.pdf')
loader2 = PyMuPDF4LLMLoader('test.pdf', mode = 'page', table_strategy = 'lines_strict') #

In [255]:
loader = PyPDFLoader('test2.pdf')
l = loader.load()

In [268]:
len(l)

2

In [267]:
l[0].metadata

{'producer': 'Microsoft® Word 2019',
 'creator': 'Microsoft® Word 2019',
 'creationdate': '2025-11-13T20:33:42+07:00',
 'author': 'Lê Nhật Linh',
 'moddate': '2025-11-13T20:33:42+07:00',
 'source': 'test2.pdf',
 'total_pages': 2,
 'page': 0,
 'page_label': '1'}

In [266]:
l[0].metadata.get('total_pages', 'N/A')

2

In [264]:
#splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=50, separators=["\n\n", "\n", " ", ""])
split_docs = splitter.split_documents(l)
len(split_docs)

8

In [276]:
type(split_docs)


list

In [285]:
l[0].metadata['total_pages']

2

In [281]:
split_docs[-1].metadata['page_label']

'2'

In [269]:
import google.generativeai as genai
genai.configure(api_key=API_KEY)
content = [i.page_content for i in split_docs]
model_embedding_name = "models/text-embedding-004"
model_chat_name = "gemini-2.5-flash"
len(genai.embed_content(model = model_embedding_name, content = content, task_type = "retrieval_document")['embedding'])

8

In [270]:
vectors = genai.embed_content(model = model_embedding_name, content = content, task_type = "retrieval_document")['embedding']

In [272]:
vectors[0]

[0.008756684,
 -0.019338332,
 -0.03211502,
 -0.052996513,
 0.017400304,
 0.012813744,
 -0.01029784,
 -0.005761692,
 -0.005992842,
 0.010425025,
 0.023659864,
 0.014177683,
 0.057146624,
 -0.020170538,
 0.011391948,
 0.0011043353,
 -0.01963092,
 0.015895445,
 -0.13172387,
 0.028376289,
 0.02577825,
 -0.016810367,
 0.011760062,
 -0.03266383,
 -0.010209398,
 -0.031906907,
 -0.015416692,
 -0.004094423,
 -0.010619443,
 -0.03662697,
 -0.014258708,
 0.041660205,
 -0.051942334,
 -0.03229625,
 -0.027595581,
 0.038766455,
 -0.032416888,
 -0.028672056,
 0.040217582,
 -0.06598755,
 -0.011776789,
 0.063771814,
 -0.009962975,
 0.03034457,
 -0.060555723,
 -0.058219817,
 -0.02494149,
 0.0996002,
 -0.04225765,
 0.05301345,
 0.05711267,
 0.0007498208,
 -0.0026944,
 0.056760143,
 0.015948553,
 -0.004886183,
 -0.018715221,
 -0.050896887,
 -0.023124482,
 -0.062391043,
 0.0540955,
 -0.07342692,
 -0.018717645,
 -0.029074501,
 0.024322774,
 0.016153837,
 0.02610098,
 -0.030759063,
 -0.06067861,
 0.03397909,
 

In [237]:
db_embed = genai.embed_content(model = model_embedding_name, content = content, task_type = "retrieval_document")['embedding']
query_embed = genai.embed_content(model = model_embedding_name, content = ["Nút dừng khẩn cấp E-Stop ở đâu?"], task_type = "retrieval_query")['embedding']

In [246]:
import numpy as np
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
for i in range(len(db_embed)):
    print(f"Doc {i} similarity: {cosine_similarity(db_embed[i], query_embed[0])}")
print(split_docs)

Doc 0 similarity: 0.2772170339387853
Doc 1 similarity: 0.30382508629912097
Doc 2 similarity: 0.3411730643626689
Doc 3 similarity: 0.3668655479881298
Doc 4 similarity: 0.30667221212626045
Doc 5 similarity: 0.2671182228282209
Doc 6 similarity: 0.3025564758672678
Doc 7 similarity: 0.25883106547390156
Doc 8 similarity: 0.31677381059811566
Doc 9 similarity: 0.4251676108587365
Doc 10 similarity: 0.4247544865629778
Doc 11 similarity: 0.294306251601062
Doc 12 similarity: 0.2827527092484571
Doc 13 similarity: 0.30006793581225
Doc 14 similarity: 0.2672007528866987
Doc 15 similarity: 0.2866910796991107
Doc 16 similarity: 0.31645503220288673
Doc 17 similarity: 0.31372161435869994
Doc 18 similarity: 0.26000774028849594
Doc 19 similarity: 0.28803600538877994
Doc 20 similarity: 0.2880897400596432
Doc 21 similarity: 0.37340632884209557
Doc 22 similarity: 0.3692930067881169
Doc 23 similarity: 0.36002362798861876
Doc 24 similarity: 0.3055975377018716
Doc 25 similarity: 0.3001305408903955
Doc 26 similari

In [222]:
model = genai.GenerativeModel(model_name='gemini-2.5-flash')

answer = model.generate_content('generate a 3 vietnamese question quiz with 4 answers based on the following context: Quy trình vận hành máy ép nhựa DENSO-X1: Bước 1: Kiểm tra an toàn. Đảm bảo nút dừng khẩn cấp (E-Stop) đang ở trạng thái mở. Đeo kính bảo hộ và găng tay chịu nhiệt. Bước 2: Khởi động nguồn. Bật công tắc chính ở phía sau')

In [229]:
display(Markdown(answer.candidates[0].content.parts[0].text))

Tuyệt vời! Dưới đây là 3 câu hỏi trắc nghiệm tiếng Việt với 4 lựa chọn dựa trên ngữ cảnh bạn đã cung cấp:

---

**Quiz: Quy trình vận hành máy ép nhựa DENSO-X1**

**Câu 1:** Theo quy trình vận hành máy ép nhựa DENSO-X1, "Bước 1" là gì?
a) Khởi động nguồn điện
b) Bật công tắc chính
c) Kiểm tra an toàn
d) Đeo kính bảo hộ và găng tay

**Câu 2:** Khi thực hiện "Bước 1: Kiểm tra an toàn", trạng thái yêu cầu của nút dừng khẩn cấp (E-Stop) là gì?
a) Đã được nhấn (activated)
b) Đang ở trạng thái đóng (closed)
c) Đang ở trạng thái mở (open/reset)
d) Không cần kiểm tra trạng thái E-Stop

**Câu 3:** Để thực hiện "Bước 2: Khởi động nguồn", hành động nào cần thiết với công tắc chính?
a) Tắt công tắc chính ở phía trước
b) Bật công tắc chính ở phía sau
c) Nhấn giữ công tắc chính trong 5 giây
d) Rút phích cắm của công tắc chính

---

**Đáp án:**
1.  **c) Kiểm tra an toàn**
2.  **c) Đang ở trạng thái mở (open/reset)**
3.  **b) Bật công tắc chính ở phía sau**

In [231]:
import google.generativeai as genai
import json
from dataclasses import dataclass

# --- CẤU HÌNH ---
# Lưu ý: Hiện tại bản chuẩn là 'gemini-1.5-flash'. 
# Nếu bạn có access bản 2.5 thì giữ nguyên, nếu lỗi 404 hãy đổi về 1.5
MODEL_NAME = 'gemini-2.5-flash' 

# Cấu hình để ép Model trả về JSON (Quan trọng nhất)
generation_config = genai.GenerationConfig(
    temperature=0.7,
    response_mime_type="application/json"
)

model = genai.GenerativeModel(
    model_name=MODEL_NAME,
    generation_config=generation_config
)

# --- 1. PHẦN ADAPTIVE PROMPT (Tạo Template) ---
def create_quiz_prompt(context_text: str, num_questions: int = 3):
    """
    Hàm này đóng vai trò như PromptTemplate của LangChain.
    Nó nhận biến đầu vào và trả về chuỗi prompt hoàn chỉnh.
    """
    # Định nghĩa cấu trúc JSON mong muốn ngay trong prompt
    json_structure = """
    [
        {
            "question": "Nội dung câu hỏi",
            "options": ["A. ...", "B. ...", "C. ...", "D. ..."],
            "answer": "Đáp án đúng (Text)",
            "explanation": "Giải thích ngắn gọn"
        }
    ]
    """
    
    # Sử dụng f-string để nhúng context vào
    return f"""
    Bạn là chuyên gia đào tạo kỹ thuật. Nhiệm vụ của bạn là tạo {num_questions} câu hỏi trắc nghiệm (Quiz) bằng Tiếng Việt dựa trên văn bản sau.
    
    VĂN BẢN NGUỒN:
    ---
    {context_text}
    ---

    YÊU CẦU:
    1. Tạo đúng {num_questions} câu hỏi.
    2. Mỗi câu hỏi có 4 đáp án lựa chọn A, B, C, D.
    3. Trả về kết quả CHỈ LÀ MỘT DANH SÁCH JSON (JSON Array) theo cấu trúc mẫu dưới đây, không thêm bất kỳ lời dẫn nào khác:
    
    CẤU TRÚC MONG MUỐN:
    {json_structure}
    """

# --- 2. THỰC THI & PARSE (Chạy thực tế) ---

# Dữ liệu đầu vào (Giả lập RAG lấy được đoạn này)
context_input = """
Quy trình vận hành máy ép nhựa DENSO-X1:
Bước 1: Kiểm tra an toàn. Đảm bảo nút dừng khẩn cấp (E-Stop) đang ở trạng thái mở. Đeo kính bảo hộ và găng tay chịu nhiệt.
Bước 2: Khởi động nguồn. Bật công tắc chính ở phía sau máy. Đợi màn hình hiển thị "Ready".
"""

# Tạo prompt động
final_prompt = create_quiz_prompt(context_input, num_questions=3)

try:
    print("⏳ Đang gọi Gemini...")
    response = model.generate_content(final_prompt)
    
    # --- 3. OUTPUT PARSER (Xử lý đầu ra) ---
    # Vì đã set mime_type="application/json", response.text chắc chắn là JSON chuẩn
    quiz_data = json.loads(response.text)
    
    print("✅ Đã parse thành công sang Python List/Dict:\n")
    
    # Demo truy cập dữ liệu như object thật
    for i, q in enumerate(quiz_data):
        print(f"Câu {i+1}: {q['question']}")
        print(f"   {q['options'][0]}")
        print(f"   {q['options'][1]}")
        print(f"   {q['options'][2]}")
        print(f"   {q['options'][3]}")
        print(f"   👉 Đáp án: {q['answer']}")
        print("-" * 30)

except json.JSONDecodeError:
    print("❌ Lỗi: Model không trả về JSON chuẩn.")
except Exception as e:
    print(f"❌ Lỗi API: {e}")

⏳ Đang gọi Gemini...
✅ Đã parse thành công sang Python List/Dict:

Câu 1: Trước khi vận hành máy ép nhựa DENSO-X1, người vận hành cần đảm bảo nút dừng khẩn cấp (E-Stop) ở trạng thái nào và phải đeo những vật dụng bảo hộ nào?
   A. E-Stop đóng; Kính bảo hộ và khẩu trang.
   B. E-Stop mở; Kính bảo hộ và găng tay chịu nhiệt.
   C. E-Stop đóng; Găng tay chịu nhiệt và mũ bảo hiểm.
   D. E-Stop mở; Khẩu trang và nút bịt tai.
   👉 Đáp án: B. E-Stop mở; Kính bảo hộ và găng tay chịu nhiệt.
------------------------------
Câu 2: Theo quy trình, sau khi bật công tắc chính ở phía sau máy, người vận hành cần đợi điều gì để biết máy đã sẵn sàng hoạt động?
   A. Đèn báo nguồn sáng xanh.
   B. Màn hình hiển thị 'Processing'.
   C. Màn hình hiển thị 'Ready'.
   D. Nghe thấy tiếng kêu 'bíp'.
   👉 Đáp án: C. Màn hình hiển thị 'Ready'.
------------------------------
Câu 3: Hành động đầu tiên trong quy trình vận hành máy ép nhựa DENSO-X1 là gì?
   A. Bật công tắc chính ở phía sau máy.
   B. Kiểm tra an toàn

In [166]:
from dataclasses import dataclass, field

@dataclass
class TextDocument:
    """Class thay thế cho LangChain Document"""
    page_content: str
    metadata: dict = field(default_factory=dict)

In [167]:
import pypdf

In [168]:
reader = pypdf.PdfReader("test.pdf")

In [181]:
from IPython.display import display, Markdown

In [187]:
reader.pages[0].extract_text(extraction_mode = 'layout')

"        The Generic Job  Satisfaction Scale:\n      Scale Development and Its Correlates\n\n                                   Scott Macdonald\n                                   Peter Maclntyre\n\n\n\n\n       ABSTRACT.   A scale on  job  satisfaction was  developed, which\n       could be   used in a wide range of occupational   groups. Aninitial item\n       pool of 44 items thought to be aspecis of .iobiatidfaction was  com-\n       pleted by a sample of     885 Ontario working adults in a wide range of\n       occupations. Factor     analysis was conducted    on the items and  a set of\n        l0 items was defined on one factor.    Cronbach's   alpha for these items\n       was .77. Average  scores on the scales were not  sighificantly different\n       between males    and females  and among       six major  oc6upational\n       groyps. The scale was significantly relateri'-to workdlace fact6rs such\n       as job  stress, boredom, isolation   and  danger of-illness or injury.\

In [188]:
display(Markdown(reader.pages[0].extract_text(extraction_mode = 'layout')))

        The Generic Job  Satisfaction Scale:
      Scale Development and Its Correlates

                                   Scott Macdonald
                                   Peter Maclntyre




       ABSTRACT.   A scale on  job  satisfaction was  developed, which
       could be   used in a wide range of occupational   groups. Aninitial item
       pool of 44 items thought to be aspecis of .iobiatidfaction was  com-
       pleted by a sample of     885 Ontario working adults in a wide range of
       occupations. Factor     analysis was conducted    on the items and  a set of
        l0 items was defined on one factor.    Cronbach's   alpha for these items
       was .77. Average  scores on the scales were not  sighificantly different
       between males    and females  and among       six major  oc6upational
       groyps. The scale was significantly relateri'-to workdlace fact6rs such
       as job  stress, boredom, isolation   and  danger of-illness or injury.
       [Article copies available from  The Haworth Document  Delivery Service:I - 8 0 0 - 3 4 2 -9 67 8. E- m ail addres s : ge I info @h owo  rth. co mJ



   Job  satisfaction is one of the most enduring  yet elusive constructs
used in the study  of industrial relations (Locke,   1976; Yuzuk,   196l).
For years   researchers have   attempted to define and measure     the
concept   ofjob   satisfaction;  however,  the scales developed to date
could be improved. In particular,      there is a need  for a valid  and
reliable scale that is short and  easily administered in the workplace.
Furthermore,  the scale should be relevant   to a wide  varietv of oc-

   Scott Macdonald, PhD,     is Scientist, Addiction Research Foundation,    100 Col-
lip Circle, Suite 200, London,    Ontario N6A  5B9. peter Maclntyre, phD, is profes-
sor, Department of Psychology,   University College of Cape Breton, p. O. Box
5300, Sydney, Nova       Scotia Bl P 6L2.
   The authors are indebted  to Samantha Wells    for her useful comments   on  drafts
of this paper.
              O 1997 by The Haworth  Press, Inc. All rights Employee Assistance Quarterly, Vol. l3(2) 1997
                                                                   reserved.            I

In [214]:
reader.metadata

{'/Producer': 'GPL Ghostscript 9.05',
 '/CreationDate': "D:20140204192605-08'00'",
 '/ModDate': "D:20140204192605-08'00'"}

In [171]:
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    print(f"--- Page {i+1} ---")
    print(text)

--- Page 1 ---
The Generic Job Satisfaction Scale:
Scale Development and Its Correlates
Scott Macdonald
Peter Maclntyre
Scott Macdonald, PhD, is Scientist, Addiction Research Foundation, 100 Col-
lip Circle, Suite 200, London, Ontario N6A 5B9. peter Maclntyre, phD, is profes-
sor, Department of Psychology, University College of Cape Breton, p. O. Box
5300, Sydney, Nova Scotia Bl P 6L2.
The authors are indebted to Samantha Wells for her useful comments on drafts
of this paper.
ABSTRACT. A scale on job satisfaction was developed, which
could be used in a wide range of occupational groups. Aninitial item
pool of 44 items thought to be aspecis of .iobiatidfaction was com-
pleted by a sample of 885 Ontario working adults in a wide range of
occupations. Factor analysis was conducted on the items and a set of
l0 items was defined on one factor. Cronbach's alpha for these items
was .77. Average scores on the scales were not sighificantly different
between males and females and among six major 

In [191]:
l = loader.load()

In [209]:
print(l[0].metadata)
print(l[0].page_content[:75])


{'producer': 'GPL Ghostscript 9.05', 'creator': 'PyPDF', 'creationdate': '2014-02-04T19:26:05-08:00', 'moddate': '2014-02-04T19:26:05-08:00', 'source': 'test.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}
The Generic Job Satisfaction Scale:
Scale Development and Its Correlates
Sc


In [211]:
import fitz
docs = fitz.open("test.pdf")

In [213]:
docs.metadata

{'format': 'PDF 1.4',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'creator': '',
 'producer': 'GPL Ghostscript 9.05',
 'creationDate': "D:20140204192605-08'00'",
 'modDate': "D:20140204192605-08'00'",
 'trapped': '',
 'encryption': None}